<a href="https://colab.research.google.com/github/agarwalagrim1910/image_generation_system_/blob/main/imageGeneration1_O.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available() and torch.cuda.device_count())
print(torch.cuda.get_device_name())

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.urg/wh1/cu118

In [ ]:
!pip install diffusers==0.21.0 transformers==4.30.2 accelerate==0.20.3

In [ ]:
!pip install safetensors==0.3.1 xformers==0.0.20 Pillow==9.5.0


In [ ]:
!pip install numpy==1.24.4 matplotlib==3.7.2 gradio==4.0.0

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import torch
import torch.nn.functional as f

In [ ]:
from diffusers import StableDiffusionPipeline
import gradio as gr
from PIL import Image
import torch.nn.functional as f
from torch import autocast
import numpy as np
from PIL import Image
import os,time,gc
from typing import List,Tuple,Optional
from datetime import datetime
from diffusers import (
    StableDiffusionPipeline,
    EulerAncestralDiscreteScheduler,
    DPMSolverMultistepScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    LMSDiscreteScheduler
)

In [ ]:
print(f'Pytorch version: {torch.__version__}')
# Add parentheses here:
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')


In [ ]:
 # ================= INSTALL (Run once in Colab) =================
# !pip install diffusers transformers accelerate gradio

# ================= IMPORTS (FIXED ALL ERRORS) =================
import torch
from torch import autocast
import time
import gc
import os
from datetime import datetime
from typing import Optional, Tuple

from PIL import Image
import gradio as gr

from diffusers import StableDiffusionPipeline
from diffusers import (
    EulerAncestralDiscreteScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    LMSDiscreteScheduler
)

# ================= GENERATOR CLASS =================
class StableDiffusionGenerator:
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5", device="auto"):
        self.device = self._setup_device(device)
        self.dtype = torch.float16 if self.device.type == "cuda" else torch.float32

        print(f"Initializing on {self.device}")
        self.pipe = self._load_pipeline(model_id)

        self.current_scheduler = "euler_a"
        self.schedulers = {
            "euler_a": ("Euler Ancestral", ""),
            "euler": ("Euler", ""),
            "ddim": ("DDIM", ""),
            "dpm_solver": ("DPM Solver", ""),
            "lms": ("LMS", "")
        }

    def _setup_device(self, device):
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        return torch.device(device)

    def _load_pipeline(self, model_id):
        pipe = StableDiffusionPipeline.from_pretrained(
            model_id,
            torch_dtype=self.dtype,
            safety_checker=None
        )

        pipe.enable_attention_slicing()

        if self.device.type == "cuda":
            pipe = pipe.to(self.device)
        else:
            pipe.enable_sequential_cpu_offload()

        return pipe

    def set_scheduler(self, name):
        scheduler_map = {
            "euler_a": EulerAncestralDiscreteScheduler,
            "euler": EulerDiscreteScheduler,
            "ddim": DDIMScheduler,
            "dpm_solver": DPMSolverMultistepScheduler,
            "lms": LMSDiscreteScheduler
        }

        if name not in scheduler_map:
            return

        self.pipe.scheduler = scheduler_map[name].from_config(
            self.pipe.scheduler.config
        )

    def generate_image(
        self,
        prompt,
        negative_prompt="",
        width=512,
        height=512,
        steps=20,
        guidance=7.5,
        seed=None,
        scheduler="euler_a"
    ):
        self.set_scheduler(scheduler)

        if seed is None:
            seed = torch.randint(0, 100000, (1,)).item()

        generator = torch.Generator(device=self.device).manual_seed(seed)

        with torch.inference_mode():
            if self.device.type == "cuda":
                with autocast("cuda"):
                    result = self.pipe(
                        prompt=prompt,
                        negative_prompt=negative_prompt,
                        width=width,
                        height=height,
                        num_inference_steps=steps,
                        guidance_scale=guidance,
                        generator=generator
                    )
            else:
                result = self.pipe(
                    prompt=prompt,
                    negative_prompt=negative_prompt,
                    width=width,
                    height=height,
                    num_inference_steps=steps,
                    guidance_scale=guidance,
                    generator=generator
                )

        return result.images[0], {"seed": seed}


# ================= UI CLASS =================
class StableDiffusionUI:
    def __init__(self):
        self.generator = None

    def initialize(self, model, device):
        self.generator = StableDiffusionGenerator(model, device)
        return "Model Loaded Successfully!"

    def generate(self, prompt):
        if self.generator is None:
            return None, "Initialize model first!"

        image, meta = self.generator.generate_image(prompt)
        return image, f"Seed: {meta['seed']}"

    def create_ui(self):
        with gr.Blocks() as demo:
            gr.Markdown("# Stable Diffusion Generator")

            model = gr.Dropdown(
                ["runwayml/stable-diffusion-v1-5"],
                value="runwayml/stable-diffusion-v1-5",
                label="Model"
            )

            device = gr.Dropdown(
                ["auto", "cuda", "cpu"],
                value="auto",
                label="Device"
            )

            init_btn = gr.Button("Initialize")
            status = gr.Textbox()

            prompt = gr.Textbox(label="Prompt")
            gen_btn = gr.Button("Generate")

            output = gr.Image()
            info = gr.Textbox()

            init_btn.click(self.initialize, [model, device], status)
            gen_btn.click(self.generate, prompt, [output, info])

        return demo


# ================= RUN =================
ui = StableDiffusionUI()
app = ui.create_ui()
app.launch()